In [60]:
from dotenv import load_dotenv
from langchain_google_genai import ChatGoogleGenerativeAI
from pydantic import BaseModel, Field
from typing import TypedDict, List, Annotated, Literal
import operator
from langchain_core.prompts import ChatPromptTemplate
from langgraph.types import Send
from langgraph.graph import  StateGraph, START, END
from langchain_groq import ChatGroq
from langchain_ollama import ChatOllama
from langchain_community.tools import DuckDuckGoSearchRun
# from duckduckgo_search import DDGS
from ddgs import DDGS
import requests
from typing import Optional
from langchain_community.tools import tool
from typing import Dict
import base64

load_dotenv()

True

In [61]:


class Task(BaseModel):
    id: int
    title: str
    goal: str = Field(
        ...,
        description="One sentence describing what the reader should be able to do/understand after this section.",
    )
    bullets: List[str] = Field(
        ...,
        min_length=3,
        max_length=5,
        description="3-5 concrete, non-overlapping subpoints to cover in this section.",
    )
    target_words: int = Field(
        ...,
        description="Target word count for this section (120-450).",
    )
    section_type: Literal[
        "intro", "core", "examples", "checklist", "common_mistakes", "conclusion"
    ] = Field(
        ...,
        description="Use 'common_mistakes' and 'examples' exactly once in the plan.",
    )
class Plan(BaseModel):
    blog_title: str = Field(..., description="A catchy and highly technical title for the blog post.")
    audience: str = Field(..., description="Who this blog is for.")
    tone: str = Field(..., description="Writing tone (e.g., practical, crisp).")
    tasks: List[Task]




class RouterDecision(BaseModel):
    """Decide if the topic requires real-time web research."""
    needs_research: bool = Field(
        description="Whether the topic requires external web search (True) or not (False)."
    )
    reason: str = Field(
        description="Brief explanation of why research is or isn't needed."
    )
    queries: List[str] = Field(default_factory=list, description="high-signal queries")


class EvidenceItem(BaseModel):
    title: str
    url: str
    published_at: Optional[str] = None  # keep if web search provides; DO NOT rely on it
    snippet: Optional[str] = None
    source: Optional[str] = None

class EvidenceWrapper(BaseModel):
    evidence: List[EvidenceItem] = Field(default_factory=list)


class ImageSpec(BaseModel):
    placeholder: str = Field(..., description="eg: [[IMAGE_1]]")
    filename: str = Field(..., description="Save under images/, e.g. qkv_flow.png")
    alt: str
    caption: str
    prompt: str = Field(..., description="Prompt to send to the image model.")
    # size: Literal["1024x1024", "1024x1536", "1536x1024"] = "1024x1024"
    # quality: Literal["low", "medium", "high"] = "medium"

class GlobalImagePlan(BaseModel):
    md_with_placeholders: str
    images: List[ImageSpec] = Field(default_factory=list)


class State(TypedDict):
    topic: str

    # routing / research
    mode: str
    needs_research: bool
    queries: List[str]
    evidence: List[EvidenceItem]
    plan: Optional[Plan]

    # workers
    sections: Annotated[List[tuple[int, str]], operator.add]  # (task_id, section_md)

    # reducer/image
    merged_md: str
    md_with_placeholders: str
    image_specs: List[ImageSpec]
    verified_images: Dict[str, str]

    final: str


In [62]:
def get_primary_llm():
    return ChatGoogleGenerativeAI(model='models/gemini-2.5-flash')

def get_secondary_llm():
    # return ChatGroq(model="llama-3.1-8b-instant")
    return ChatOllama(model="llama3")

In [63]:
def router_node(state : State):
    topic = state["topic"]

    system_prompt = """You are an intelligent routing module for an AI research agent.\nYour objective is to analyze a user's topic and determine if external web research is required before generating a response.\n
    If needs_research=true:\n
    - Output 3-10 high-signal queries.
    - Queries should be scoped and specific (avoid generic queries like just "AI" or "LLM").
    - If user asked for "last week/this week/latest", reflect that constraint IN THE QUERIES..
    """

    prompt = ChatPromptTemplate.from_messages([
        ("system", system_prompt),
        ("human", "Topic: {topic}")
    ])

    chain = prompt | get_secondary_llm().with_structured_output(RouterDecision)
    result = chain.invoke({"topic": topic})

    return {
        "router_decision": result
    }


def route_next(state: State) -> str:
    return "research" if state["router_decision"].needs_research else "orchestrator"

In [64]:
# web search based on query and url


def _searxng_search(query: str, max_results: int = 5) -> List[dict]:
    """Helper function to fetch and normalize results from local SearxNG."""
    url = "http://localhost:8080/search"

    params = {
        "q": query,
        "format": "json",
        "engines": "google,bing,duckduckgo", # Add/remove engines as needed
        "language": "en-US"
    }

    try:
        response = requests.get(url=url, params=params, timeout=10)
        response.raise_for_status()
        data = response.json()

        normalized: List[dict] = []

        for r in data.get("results", [])[:max_results]:
            normalized.append({
                "title": r.get("title") or "",
                "url": r.get("url") or "",
                "snippet": r.get("content") or r.get("snippet") or "",
                "published_at": r.get("published_date"), 
                #SearxNG uses this for some engines
                "source": r.get("engine"),
            })
        return normalized
    except Exception as e:
        print(f"Error calling SearxNG: {e}")
        return []
    
@tool
def search_specific_site(query: str, site: str, max_results: int = 5) -> List[dict]:
    """
    Search for information within a specific website only.
    Use this when you have a high-quality source (like docs.python.org) 
    and need to find specific details within it.
    
    Args:
        query: The search term or question.
        site: The domain to search (e.g., 'wikipedia.org', 'github.com').
        max_results: How many results to return.
    """
    full_query = f"site:{site} {query}"
    results = _searxng_search(full_query, max_results=max_results)

    if not results:
        return "No results found for this site search."
            
    # Convert the list of dicts into a readable string for the LLM
    formatted_results = []
    for r in results:
        formatted_results.append(f"Title: {r['title']}\nSnippet: {r['snippet']}\nURL: {r['url']}")
        
    return "\n\n---\n\n".join(formatted_results)


In [65]:
def research_node(state : State) -> dict:
    router_decision = state["router_decision"]
    queries = (router_decision.get("queries", []) or [])
    system_message = """You are a Specialized Research Extraction Agent. 

    Your goal is to transform raw, messy search engine results into a structured 'Evidence Pack'. 

    ### GUIDELINES:
    1. **Veracity First**: Only extract information that is explicitly stated in the snippets. Do not hallucinate details or "fill in the blanks."
    2. **Signal-to-Noise**: Ignore results that are clearly advertisements, cookie consent text, or irrelevant SEO filler.
    3. **Temporal Awareness**: If multiple results conflict, prioritize the one with the most recent 'published_at' date or the most authoritative 'source'.
    4. **Snippet Preservation**: Keep the 'snippet' meaningful. Do not summarize it so much that the specific data points (numbers, names, specs) are lost.
    5. **No Commentary**: Do not explain your choices. Simply output the filtered EvidenceItem list.

    ### OUTPUT EXPECTATION:
    You must return a valid EvidencePack. Each EvidenceItem must have a valid URL. If no relevant results are found, return an empty list."""

    raw_results: List[dict] = []

    for q in queries:
        raw_results.extend(_searxng_search(q, max_results=3))

    if not raw_results:
        return {"evidence": []}
    
    exctractor = get_secondary_llm().with_structured_output(EvidenceWrapper)
    pack = exctractor.invoke([
        ("system", system_message),
        ("human", f"Raw results:\n{raw_results}")
    ])

    dedup = {}
    for e in pack.evidence:
        if e.url:
            dedup[e.url] = e

    return {"evidence": list(dedup.values())} 

In [66]:
def orchestrator_node(state : State) -> dict:
    planner = get_primary_llm().with_structured_output(Plan)
    orchestrator_system_message = """You are a Master Content Architect and Editor-in-Chief.
    Your goal is to create a detailed, high-signal execution plan for a technical blog post.

    ### YOUR INPUTS:
    1. **Topic**: The core subject.
    2. **Scout Evidence**: Brief snippets from the web to ground your planning in current reality (e.g., current versions, recent news, existing controversies).

    ### YOUR MISSION:
    - **Structure**: Break the topic into logical, non-overlapping tasks.
    - **Precision**: For each task, write a 'goal' and 'bullets' that are so specific that a worker with no context could write a perfect section.
    - **Research Guidance**: Provide 2-3 specific search queries for the workers to use for deep-diving into their specific section.
    - **Variety**: Ensure the 'section_type' distribution follows the requirements (using 'examples' and 'common_mistakes' exactly once).

    ### CRITICAL RULES:
    - If the 'Scout Evidence' mentions specific version numbers (e.g., Python 3.13) or dates, ensure the Plan reflects these.
    - Do not create more than 5-6 tasks total to ensure the blog remains focused.
    - Ensure the 'target_words' for the entire blog totals between 1000-1500 words."""

    evidence = state.get("evidence", [])
    evidence_str = ""

    if evidence:
        for i,e in enumerate(evidence[:10]):
            evidence_str += f"[{i+1}] {e.title}\n   Snippet: {e.snippet}\n   URL: {e.url}\n\n"
    else:
        evidence_str = "No recent search evidence found. Plan based on general knowledge."

    
    plan = planner.invoke([
        ("system", orchestrator_system_message),
        ("human", f"Topic: {state["topic"]}\nScout Evidence:\n{evidence_str}")
    ])

    print(f"📋 [Orchestrator] Plan generated with {len(plan.tasks)} sections.")
    return {"plan": plan}

In [67]:
def fanout(state: State):
    """
    This function reads the plan and spawns parallel workers.
    """
    plan = state.get("plan")
    if not plan or not plan.tasks:
        return [] # Or go to END
    

    return [
        Send("worker", {
            "task": task,
            "topic": state["topic"],
            "plan": plan,
            "evidence": state.get("evidence", [])
        })
        for task in plan.tasks
    ]

In [68]:
def clean(text: str) -> str:
    if not text: return "---"
    # Remove pipes and newlines so the Markdown table doesn't break
    return text.replace("|", "-").replace("\n", " ").strip()


def worker(payload: dict) -> dict:
    plan = Plan(**payload["plan"])
    task = Task(**payload["task"])
    topic = payload.get("topic", "")
    evidence = [EvidenceItem(**e) for e in payload.get("evidence", [])]

    bullets_text = "\n- " + "\n- ".join(task.bullets)
    
    evidence_header = "| Title | URL | Date | Snippet |"
    evidence_divider = "|-------|-----|------|---------|"
    evidence_table = ""

    if evidence:
        rows = [
            f"| {clean(e.title)} | {e.url} | {clean(e.published_at)} | {clean(e.snippet)} |"
            for e in evidence[:20]
        ]
        evidence_table = "\n".join([evidence_header, evidence_divider] + rows)
    else:
        evidence_table = "No evidence available."


    system_prompt = f"""You are a professional Technical Writer. 
    Your goal is to write the '{task.section_type}' section of a blog post titled: "{plan.blog_title}".
    
    WRITING STYLE:
    - Audience: {plan.audience}
    - Tone: {plan.tone}
    - Standards: Use clean Markdown. Be precise, technical, and avoid fluff.
    """
    
    human_message = f"""Please write the following section for our blog about '{topic}'.

    SECTION TITLE: {task.title}
    GOAL: {task.goal}
    
    KEY POINTS TO COVER:
    {bullets_text}

    RESEARCH EVIDENCE TO USE:
    {evidence_table}

    REQUIREMENTS:
    - Length: Approximately {task.target_words} words.
    - Format: Use Markdown headers (##) and bolding for key terms.
    - If there is a URL in the evidence that is highly relevant, cite it.
    """
    tools = [search_specific_site]
    llm_with_tools = get_primary_llm().bind_tools(tools)

    response = llm_with_tools.invoke([
        ("system", system_prompt),
        ("human", human_message)
    ])

    # Return a tuple with (id, content) so the Reducer can sort them correctly
    return {"sections": [(task.id, response.content)]}

In [69]:
def merge_node(state: State) -> dict:
    plan = state["plan"]

    ordered_sections = [md for _, md in sorted(state["sections"], key=lambda x: x[0])]
    body = "\n\n".join(ordered_sections).strip()

    merged_md = f"# {plan.blog_title}\n\n{body}\n"
    return {"merged_md": merged_md}

In [70]:
def image_planner_node(state: State) -> dict:
    planner = get_primary_llm().with_structured_output(GlobalImagePlan)

    system_message = """You are a Technical Visual Content Strategist. 
    Analyze the blog Markdown and identify where visual aids (diagrams, flowcharts, or technical photos) will add value.

    ACTION:
    1. Insert placeholders like [[IMAGE_1]], [[IMAGE_2]] directly into the text where they provide the most impact.
    2. For every placeholder, create a detailed ImageSpec.

    ### QUERY GENERATION RULES FOR SEARXNG:
    - The 'prompt' field MUST be a search-engine-optimized query.
    - Use technical keywords, not descriptive prose. (e.g., Use "Python JIT compiler architecture diagram" instead of "A drawing showing how a JIT compiler works").
    - Add qualifiers like "diagram", "flowchart", "architecture", or "screenshot" to the query to ensure professional results.
    - Avoid aesthetic words like "beautiful", "high-quality", or "cinematic" as they pollute search results.

    ### SECTION CONTEXT:
    - Ensure the image matches the 'blog_kind' ({state['plan'].blog_kind}). 
    - If it's a 'tutorial', prioritize screenshots or code-flow diagrams.
    - If it's a 'system_design', prioritize architectural patterns."""

    image_plan = planner.invoke([
        ("system", system_message),
        ("human", f"Blog kind: {state["plan"].blog_kind}\nTopic: {state['topic']}\nBlog Content:\n{state['merged_md']}")
    ])

    return {
        "md_with_placeholders": image_plan.md_with_placeholders,
        "image_specs": image_plan.images # Store these for the Generator node
    }

In [71]:


def _searxng_image_search(query: str, max_results: int = 3) -> str:
    """Fetches top 3 image URLs and their metadata from SearxNG."""

    url = "http://localhost:8080/search"
    params = {
        "q": query,
        "format": "json",
        "categories": "images",
        "engines": "google images,bing images"
    }
    try:
        response = requests.get(url, params=params, timeout=10)
        return response.json().get("results", [])[:3]
    except:
        return []
    

def _download_image_as_base64(url: str) -> str:
    """Downloads an image and converts it to base64 for the Vision LLM."""
    try:
        res = requests.get(url, timeout=5)
        if res.status_code == 200:
            return base64.b64encode(res.content).decode("utf-8")
    except:
        return None

In [72]:
def image_verification_node(state: State) -> dict:
    specs: List[ImageSpec] = state.get("image_specs", [])
    llm = get_secondary_llm()
    
    # This dictionary will store our final choices
    verified_images = {}

    for spec in specs:
        # 1. Fetch candidates from SearxNG
        candidates = _searxng_image_search(spec.prompt) # This returns the List[dict]
        
        if not candidates:
            continue

        # 2. Build the metadata string for the LLM to judge
        metadata_summary = "\n".join([
            f"ID {i}: Title: {c['title']} | Source: {c['source']}"
            for i, c in enumerate(candidates)
        ])

        # 3. Ask LLM to pick the index
        prompt = f"""Target Image: {spec.alt}
        {metadata_summary}
        
        Which ID is the most relevant and high-quality technical image for this topic? 
        Return ONLY the number."""

        try:
            response = llm.invoke(prompt).content.strip()
            # Basic cleaning in case LLM adds a period or extra text
            best_index = int(''.join(filter(str.isdigit, response)))
            
            if 0 <= best_index < len(candidates):
                best_url = candidates[best_index]['img_src']
                # Store the winner in our map
                verified_images[spec.placeholder] = best_url
        except Exception as e:
            print(f"Error verifying image {spec.placeholder}: {e}\nFallback to default image")
            if(len(candidates) > 0):
                verified_images[spec.placeholder] = candidates[0]['img_src']

    # We return the dictionary, which LangGraph saves to the state
    return {"verified_images": verified_images}

In [ ]:
def image_placement_node(state: State) -> dict:
    current_md = state.get("md_with_placeholders", "")
    verified_images = state.get("verified_images", {})
    specs = state.get("image_specs", [])

    # FIX: Initialize final_md before the loop
    final_md = current_md 

    for spec in specs:
        placeholder = spec.placeholder
        if placeholder in verified_images:
            url = verified_images[placeholder]
            image_markdown = f"\n\n![{spec.alt}]({url})\n*Figure: {spec.caption}*\n\n"
            final_md = final_md.replace(placeholder, image_markdown)
        else:
            # Cleanup stray placeholders
            final_md = final_md.replace(placeholder, "")
    
    return {"final": final_md} # Ensure this matches your State['final'] key